# Aşama 6: Stacking

**Fikir:** Tek tek zayıf olan kaynakların sinyallerini **tek bir testte toplamak**. Her kaynak 2–3σ olabilir; hepsi gerçekten nötrino yayıyorsa birlikte daha güçlü bir sinyal verir.

Likelihood aynı, tek fark sinyal yoğunluğu: artık bir olay "dört kaynaktan herhangi birine yakınsa" sinyal gibi görünüyor. Kaynaklara, dedektörün o yöndeki hassasiyetine göre ağırlık veriyoruz (Aşama 5'teki etkin alan hesabı).

In [ ]:
import sys; sys.path.append('../src')
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import icecube_ps as ps
importlib.reload(ps)
assert float(ps.SURUM) >= float('7.0'), f"src/icecube_ps.py eski sürüm ({getattr(ps, 'SURUM', 'yok')}). Yenisini kopyala, kernel'i yeniden başlat."
try:
    from tqdm.auto import tqdm          # ilerleme çubuğu
except ImportError:
    tqdm = None

pd.set_option('display.float_format', '{:.3g}'.format)
DATA, RESULTS = Path('../data'), Path('../results')
ev = ps.veri_yukle(DATA)
B = ps.ArkaPlan(ev['dec'].values)
print('icecube_ps', ps.SURUM, '| olay:', f'{len(ev):,}')

In [ ]:
alanlar = ps.etkin_alanlari_yukle(DATA)
sureler = ps.canli_sureler_yukle(DATA)
agirlik = [ps.aki_basina_olay(dk, 2.0, alanlar, sureler) for _, dk in ps.KAYNAKLAR.values()]
print(pd.Series(np.array(agirlik) / sum(agirlik), index=list(ps.KAYNAKLAR), name='ağırlık'))

S = ps.StackingLikelihood(ev, B, ps.KAYNAKLAR, agirlik)
ns, ts = S.fit()
print(f"\nStacking:  nₛ = {ns:.1f}   TS = {ts:.2f}")

## Anlamlılık

Yine RA karıştırma. ⏱️ 5000 deneme, birkaç dakika.

In [ ]:
dosya = RESULTS / 'ts_arka_stacking.npy'
if dosya.exists():
    ts_arka = np.load(dosya)
else:
    ts_arka = S.arka_plan_denemeleri(5000, np.random.default_rng(6), ilerleme=tqdm)
    np.save(dosya, ts_arka)

p_mc, p_w = ps.p_degeri(ts, ts_arka)
print(f"Monte Carlo: {ps.sigma(p_mc):.2f}σ   |   Wilks: {ps.sigma(p_w):.2f}σ")

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(ts_arka, bins=60, color='#7a8fb8', label='Karıştırılmış gökyüzleri')
ax.axvline(ts, color='#d1495b', lw=2.5, label=f'Gözlenen TS = {ts:.1f}')
ax.set_yscale('log'); ax.set_xlabel('TS'); ax.legend()
ax.set_title(f'4 kaynak stacking: {ps.sigma(p_w):.1f}σ (Wilks)')
plt.savefig(RESULTS / 'asama6_stacking.png', dpi=150, bbox_inches='tight')
plt.show()

pd.DataFrame([dict(test='stacking (4 kaynak)', ns=ns, TS=ts,
                   sigma_MC=ps.sigma(p_mc), sigma_Wilks=ps.sigma(p_w))]).to_csv(
    RESULTS / 'asama6_stacking.csv', index=False)

## ⚠️ Dürüst not

Bu dört kaynağı IceCube fazlalık bulduğu için seçtik. Sonucu bilerek seçilmiş kaynakları birleştirmek anlamlılığı **şişirir**; bu sayı bir keşif iddiası olamaz, yöntemin gösterimidir. Gerçek bir stacking analizi kaynak listesini veriye bakmadan önce belirler (örneğin "en parlak 100 gama ışını kaynağı"). Aşama 10'da Fermi-LAT kataloğuyla bunu yapacağız.

Tarafsız soru şu: "Kaynak listesi olmadan, gökyüzünün **neresinde** fazlalık var?" Bu Aşama 7.